<a href="https://colab.research.google.com/github/rohithsai199/coastal_seven_tasks/blob/main/CLI_task_manager_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install PostgreSQL and driver
!apt-get update -qq > /dev/null
!apt-get install -y postgresql postgresql-contrib > /dev/null
!pip install -q psycopg2-binary

# 2. Start PostgreSQL service
!service postgresql start

# 3. Setup database credentials and privileges
!sudo -u postgres psql -c "CREATE USER taskmaster WITH PASSWORD 'securepass123';"
!sudo -u postgres psql -c "CREATE DATABASE taskdb OWNER taskmaster;"
!sudo -u postgres psql -c "GRANT ALL PRIVILEGES ON DATABASE taskdb TO taskmaster;"

print("PostgreSQL server is running and ready for connections!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 45.3 MB/s eta 0:00:00
[ OK ]
CREATE ROLE
CREATE DATABASE
GRANT
PostgreSQL server is running and ready for connections!


In [2]:
import psycopg2
from psycopg2.extras import RealDictCursor

DB_CONFIG = {
    "dbname": "taskdb",
    "user": "taskmaster",
    "password": "securepass123",
    "host": "localhost",
    "port": "5432"
}

def init_db():
    schema_sql = """
    -- Users Table (Relational Entity)
    CREATE TABLE IF NOT EXISTS users (
        user_id SERIAL PRIMARY KEY,
        username VARCHAR(50) UNIQUE NOT NULL,
        email VARCHAR(100) UNIQUE NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Tasks Table (Foreign Key relationship)
    CREATE TABLE IF NOT EXISTS tasks (
        task_id SERIAL PRIMARY KEY,
        user_id INT NOT NULL REFERENCES users(user_id) ON DELETE CASCADE,
        title VARCHAR(200) NOT NULL,
        status VARCHAR(20) DEFAULT 'pending' CHECK (status IN ('pending', 'in_progress', 'completed')),
        priority INT DEFAULT 1 CHECK (priority BETWEEN 1 AND 5),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Indexes to optimize JOINs and filtering performance
    CREATE INDEX IF NOT EXISTS idx_tasks_user_id ON tasks(user_id);
    CREATE INDEX IF NOT EXISTS idx_tasks_status ON tasks(status);
    """
    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            cur.execute(schema_sql)
            conn.commit()
            print("Schema, Keys, and Indexes initialized successfully.")

init_db()

Schema, Keys, and Indexes initialized successfully.


In [3]:
class Task:
    """Domain model representing a single Task entity."""
    def __init__(self, title: str, user_id: int, status: str = "pending", priority: int = 1, task_id: int = None):
        self.task_id = task_id
        self.user_id = user_id
        self.title = title
        self.status = status
        self.priority = priority

    def __str__(self):
        return f"[ID: {self.task_id}] {self.title} | Status: {self.status} | Priority: {self.priority}"


class TaskManager:
    """Manages database persistence, queries, JOINs, and aggregates using psycopg2."""

    def __init__(self, db_config: dict):
        self.db_config = db_config

    def _get_connection(self):
        return psycopg2.connect(**self.db_config)

    def register_user(self, username: str, email: str) -> int:
        """Create a new user or fetch existing user ID."""
        sql = """
        INSERT INTO users (username, email)
        VALUES (%s, %s)
        ON CONFLICT (email) DO UPDATE SET username = EXCLUDED.username
        RETURNING user_id;
        """
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, (username, email))
                user_id = cur.fetchone()[0]
                conn.commit()
                return user_id

    def add_task(self, task: Task) -> Task:
        """Persist a new task to PostgreSQL."""
        sql = """
        INSERT INTO tasks (user_id, title, status, priority)
        VALUES (%s, %s, %s, %s)
        RETURNING task_id;
        """
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, (task.user_id, task.title, task.status, task.priority))
                task.task_id = cur.fetchone()[0]
                conn.commit()
                return task

    def update_task_status(self, task_id: int, status: str) -> bool:
        """Update status for a specific task."""
        sql = "UPDATE tasks SET status = %s WHERE task_id = %s;"
        with self._get_connection() as conn:
            with conn.cursor() as cur:
                cur.execute(sql, (status, task_id))
                conn.commit()
                return cur.rowcount > 0

    def list_user_tasks(self, user_id: int):
        """Retrieve tasks using an INNER JOIN between users and tasks."""
        sql = """
        SELECT
            u.username,
            t.task_id,
            t.title,
            t.status,
            t.priority
        FROM users u
        INNER JOIN tasks t ON u.user_id = t.user_id
        WHERE u.user_id = %s
        ORDER BY t.priority DESC;
        """
        with self._get_connection() as conn:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(sql, (user_id,))
                return cur.fetchall()

    def get_summary_statistics(self):
        """
        Demonstrates Aggregate Functions (COUNT, AVG), GROUP BY, and HAVING.
        Calculates aggregate stats for users with at least 1 task.
        """
        sql = """
        SELECT
            u.username,
            COUNT(t.task_id) AS total_tasks,
            COUNT(CASE WHEN t.status = 'completed' THEN 1 END) AS completed_tasks,
            COUNT(CASE WHEN t.status = 'pending' THEN 1 END) AS pending_tasks,
            ROUND(AVG(t.priority), 2) AS avg_priority
        FROM users u
        LEFT JOIN tasks t ON u.user_id = t.user_id
        GROUP BY u.user_id, u.username
        HAVING COUNT(t.task_id) > 0
        ORDER BY total_tasks DESC;
        """
        with self._get_connection() as conn:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(sql,)
                return cur.fetchall()

In [4]:
def run_cli_app():
    manager = TaskManager(DB_CONFIG)

    print("==========================================")
    print("     WELCOME TO CLI TASK MANAGER DB       ")
    print("==========================================")

    username = input("Enter your username: ").strip()
    email = input("Enter your email: ").strip()

    current_user_id = manager.register_user(username, email)
    print(f"\nAuthenticated as: {username} (User ID: {current_user_id})\n")

    while True:
        print("\n--- MENU ---")
        print("1. Add New Task")
        print("2. View My Tasks (INNER JOIN)")
        print("3. Update Task Status")
        print("4. View All Users Summary (GROUP BY / HAVING)")
        print("5. Exit")

        choice = input("Select an option (1-5): ").strip()

        if choice == '1':
            title = input("Enter task title: ").strip()
            priority = int(input("Enter priority (1-5): ").strip() or 1)
            new_task = Task(title=title, user_id=current_user_id, priority=priority)
            created = manager.add_task(new_task)
            print(f"Task successfully created: {created}")

        elif choice == '2':
            tasks = manager.list_user_tasks(current_user_id)
            print(f"\n--- Tasks for {username} ---")
            if not tasks:
                print("No tasks found.")
            for t in tasks:
                print(f"[{t['task_id']}] {t['title']} | Status: {t['status']} | Priority: {t['priority']}")

        elif choice == '3':
            task_id = int(input("Enter Task ID to update: ").strip())
            print("Status options: pending, in_progress, completed")
            status = input("Enter new status: ").strip().lower()
            if manager.update_task_status(task_id, status):
                print("Task status updated successfully!")
            else:
                print("Task ID not found.")

        elif choice == '4':
            stats = manager.get_summary_statistics()
            print("\n--- System User Productivity Summary ---")
            for row in stats:
                print(f"User: {row['username']} | Total: {row['total_tasks']} | Completed: {row['completed_tasks']} | Pending: {row['pending_tasks']} | Avg Priority: {row['avg_priority']}")

        elif choice == '5':
            print("Exiting CLI application. Goodbye!")
            break
        else:
            print("Invalid choice, please select 1-5.")

# Run the CLI app in Colab
run_cli_app()

     WELCOME TO CLI TASK MANAGER DB       
Enter your username: rohith
Enter your email: guntururohithsai@gmail.com

Authenticated as: rohith (User ID: 1)


--- MENU ---
1. Add New Task
2. View My Tasks (INNER JOIN)
3. Update Task Status
4. View All Users Summary (GROUP BY / HAVING)
5. Exit
Select an option (1-5): 3
Enter Task ID to update: 1
Status options: pending, in_progress, completed
Enter new status: 5
Task ID not found.

--- MENU ---
1. Add New Task
2. View My Tasks (INNER JOIN)
3. Update Task Status
4. View All Users Summary (GROUP BY / HAVING)
5. Exit
Select an option (1-5): 5
Exiting CLI application. Goodbye!
